## Escraping un logement

In [1]:
import requests 
import lxml
from bs4 import BeautifulSoup as bs
import pandas as pd

url = 'https://trouverunlogement.lescrous.fr/tools/42/accommodations/1606'

request_text = requests.get(url, headers={"User-Agent": "Python for data science tutorial"}).content

page = bs(request_text, "lxml") 
print(page.prettify()[:500])

texte = page.find('ul', class_="fr-raw-list")


<!DOCTYPE html>
<html lang="fr">
 <head>
  <meta charset="utf-8"/>
  <link href="../../../favicon.ico" rel="icon"/>
  <meta content="width=device-width" name="viewport"/>
  <link href="../../../_app/immutable/assets/_layout-ef99e2ee.css" nonce="Lk+ONcsMRejbFTwAtsHPGA==" rel="stylesheet"/>
  <link href="../../../_app/immutable/assets/PageTree-b358c7ce.css" nonce="Lk+ONcsMRejbFTwAtsHPGA==" rel="stylesheet"/>
  <link href="../../../_app/immutable/assets/_page-430ef9ac.css" nonce="Lk+ONcsMRejbFTwAts


In [2]:
adresse = page.find('p', class_="fr-text--lead")
adresse = adresse.text.split()
code = adresse[-2]
ville = adresse[-1]

In [3]:
rows = texte.find_all('li')

lignes = [ligne.text.split() for ligne in rows][:10]

adresse = page.find('p', class_="fr-text--lead")
adresse = adresse.text.split()
code_postal = dict(code = adresse[-2], ville = adresse[-1])

dico = {i[0] : ' '.join(i[2:]) for i in lignes}
dico.update(code_postal)


df = pd.DataFrame(dico, index=[0])
df

,Superficie,Équipements,Lits,Bus,Parking,Stationnement,Description,Une,Salle,WC,code,ville
0,"21,77 m²","WC, Douche, Evier + plaque, Frigo",1 lit simple,5 minutes à pied De l'arrêt Charles Perrens : ...,Un parking avec nombre de places limité / disp...,gratuit,LOGEMENT DISPONIBLE A COMPTER DU 16/01/2026 Im...,"comprenant un petit lit, un bureau, une table,...",bain avec baignoire.,la salle de bain.,33000,BORDEAUX


# ESCRAPPER TOUTE LA PAGE

In [4]:
import requests 
import lxml
from bs4 import BeautifulSoup as bs
import pandas as pd

url_annonces = 'https://trouverunlogement.lescrous.fr/tools/42/search?bounds=-0.6386987_44.9161806_-0.5336838_44.8107826'

request_annonces = requests.get(url_annonces, headers={"User-Agent": "Python for data science tutorial"}).content

page_annonces = bs(request_annonces, "lxml") 

texte_annonces = [lien.find('a') for lien in page_annonces.find_all('h3')]
liens = [lien.get('href') for lien in texte_annonces]
liens

['/tools/42/accommodations/1503']

In [5]:
dico_page = {}
for i in liens :
    dico_2 = {}
    url = 'https://trouverunlogement.lescrous.fr'+i

    request_text = requests.get(url, headers={"User-Agent": "Python for data science tutorial"}).content
    page = bs(request_text, "lxml") 

    adresse = page.find('p', class_="fr-text--lead")
    adresse = adresse.text.split()
    code_postal = dict(code = adresse[-2], ville = adresse[-1])
    dico_2.update(code_postal)
    
    texte = page.find('ul', class_="fr-raw-list")
    rows = texte.find_all('li')
    lignes = [ligne.text.split() for ligne in rows][:10]
    
    for j in lignes :
        elem = {f"{j[0]}" : f"{' '.join(j[2:]).replace(': ','')}"}
        dico_2.update(elem)
    dico_page[url]= dico_2

dico_page


{'https://trouverunlogement.lescrous.fr/tools/42/accommodations/1503': {'code': '33000',
  'ville': 'BORDEAUX',
  'Superficie': '12 m²',
  'Type': 'cohabitation Individuel',
  'Loyer': 'Individuel 262 €',
  'Individuel': '262 €',
  'Avance': 'du premier mois de loyer 100 €',
  'Équipements': 'Frigo',
  'Lits': '1 lit simple',
  'Bus': '500 mètres Arrêt Navarre - Liane 4 et 11',
  'Tramway': '500 mètres Tram A - Arrêt Palais de justice (400m) et Hôtel de police (600 m)'}}

In [6]:
df_dict = pd.DataFrame(dico_page)
h = df_dict.T
h

,code,ville,Superficie,Type,Loyer,Individuel,Avance,Équipements,Lits,Bus,Tramway
https://trouverunlogement.lescrous.fr/tools/42/accommodations/1503,33000,BORDEAUX,12 m²,cohabitation Individuel,Individuel 262 €,262 €,du premier mois de loyer 100 €,Frigo,1 lit simple,500 mètres Arrêt Navarre - Liane 4 et 11,500 mètres Tram A - Arrêt Palais de justice (4...


# ESCRAPPER TOUT LE SITE

In [ ]:
import requests 
import lxml
from bs4 import BeautifulSoup as bs
import pandas as pd

dico_url = {}


for i in range(1, 4001) :
    logements = {}
    try : 
        url = 'https://trouverunlogement.lescrous.fr/tools/42/accommodations/'+str(i)
        
        request_text = requests.get(url, headers={"User-Agent": "Python for data science tutorial"}).content
        page = bs(request_text, "lxml") 

        adresse = page.find('p', class_="fr-text--lead")
        adresse = adresse.text.split()
        Adresse = ' '.join(adresse[:-2]).replace(',', '')
        code_postal = dict(code = adresse[-2], ville = adresse[-1], Adresse = Adresse)
        logements.update(code_postal)
    
        texte = page.find('ul', class_="fr-raw-list")
        rows = texte.find_all('li')
        
        lignes = [ligne.text.split() for ligne in rows]

        for j in lignes :
            elem = {f"{j[0]}" : f"{' '.join(j[2:]).replace(': ','')}"}
            logements.update(elem)
            
        dico_url[url]= logements

    except : AttributeError
    continue

df_logements = pd.DataFrame(dico_url)
df_logements = df_logements.T

df_logements.to_csv('logements_étudiants.csv', index = False, encoding = 'utf-8-sig')